# SMS / Email Spam Classifier
This notebook demonstrates the pipeline for text classification using Multinomial Naive Bayes.

In [ ]:
import pandas as pd
import string
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score
import pickle

nltk.download('punkt', quiet=True)

In [ ]:
# Load Data
df = pd.read_csv('spam.csv', encoding='latin-1')[['v1', 'v2']].copy()
df.columns = ['label', 'text']
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

In [ ]:
# Preprocessing
ps = PorterStemmer()

def transform_text(text):
    text = text.lower()
    return ' '.join([ps.stem(w) for w in nltk.word_tokenize(text) if w.isalnum()])

df['transformed_text'] = df['text'].apply(transform_text)
df.head()

In [ ]:
# Vectorization and Train-Test Split
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['label_enc'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

In [ ]:
# Training Multinomial Naive Bayes
# MNB gives 100% precision, ensuring no ham messages are classified as spam.
mnb = MultinomialNB()
mnb.fit(X_train, y_train)

y_pred = mnb.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))